# Group Homework 2 · GoMart deliveries
**Data Visualization · UEH · Session 2** · team notebook

| | |
|---|---|
| **Team** | *team name* |
| **Status** | Formative: written feedback with the rubric in the brief (first step of the group project) |
| **Submit** | File ▸ Download ▸ .ipynb as `GH2_TeamName.ipynb`, within 7 days on the LMS |

Read the brief *Group Homework 2 · GoMart* first. All data is simulated.

## Memo to the operations manager (Task 6, write this last)
**Big Idea:** *one complete sentence: your point and what is at stake.*

*At most 120 words. Put your explanatory chart in the cell below.*

In [13]:
# Task 6: your explanatory chart goes here (copy it from Task 5 and make it explanatory)

## Task 0 · Team card
**Team name:** *(to agree)*

| Student ID | Name | Role |
|---|---|---|
| 31221020403 | Phạm Tuấn Anh | |
| 31241027205 | Trần Nhã Đoan | |
| 31221021575 | Nguyễn Hoàng Minh | Leader |
| 31221021576 | Phạm Quang Minh | |
| 31241021154 | Nguyễn Minh Nguyệt | |
| 31241026554 | Trần Nguyễn Thanh Thảo | |

**Candidate project datasets**

| # | Dataset and source | Data abstraction | Two questions as tasks (action + target) |
|---|---|---|---|
| 1 | **World Development Indicators**, World Bank Open Data ([data.worldbank.org](https://data.worldbank.org/indicator/IT.NET.USER.ZS)) | Multidimensional table, static: keys *country* + *year*; values are quantitative indicators such as internet users (% of population) and GDP per capita. | *Summarize* the *trend* of internet use in Viet Nam and its ASEAN neighbours, 2000–2024. · *Discover* the *correlation* between GDP per capita and internet use across ASEAN countries. |
| 2 | **Brazilian E-Commerce Public Dataset by Olist**, Kaggle ([kaggle.com/datasets/olistbr/brazilian-ecommerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)) | Several flat tables linked by keys (*order_id*, *customer_id*), static, about 99,000 orders 2016–2018: delivery timestamps (quantitative), review score 1–5 (ordinal), customer state (categorical). | *Discover* the *dependency* of the review score on late delivery. · *Compare* the *distribution* of delivery time across customer states. |
| 3 | **Provincial statistics of Viet Nam**, National Statistics Office, formerly GSO ([nso.gov.vn](https://www.nso.gov.vn)) | Multidimensional table, static: keys *province* + *year*; values such as population and monthly income per capita (quantitative); region (categorical). Can be joined to province boundaries (geometry) for a map. | *Compare* the *extremes* of income per capita across provinces. · *Summarize* the *trend* of income by region. Check first: provinces were merged in 2025, so tables may use different province lists. |

## Setup
In the repo the data is read from `data/raw`. In Colab, upload `gomart_orders_2025.csv` and `gomart_districts.csv` first (folder icon on the left, or `files.upload()`).

In [14]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# In the repo the files are in data/raw; in Colab, upload both files next to the notebook
DATA_DIR = next((p for p in [Path("../../data/raw"), Path("data/raw"), Path(".")]
                 if (p / "gomart_orders_2025.csv").exists()), None)
assert DATA_DIR is not None, "Upload gomart_orders_2025.csv and gomart_districts.csv first"
# The clean CSV (Task 4) is saved next to this file, or in the working folder in Colab
HERE = Path(__file__).parent if "__file__" in globals() else Path(".")
raw = pd.read_csv(DATA_DIR / "gomart_orders_2025.csv")
districts = pd.read_csv(DATA_DIR / "gomart_districts.csv")
print("orders:", raw.shape, "| districts:", districts.shape)

AssertionError: Upload gomart_orders_2025.csv and gomart_districts.csv first

## Task 1 · Look
Run the first-look checklist. Then fill the table of problems below.

In [ ]:
# 1 · How big is the file, what does a row look like, is each column the type we expect?
print(raw.shape)
print(raw.head().to_string())
raw.info()
print(raw.nunique().to_string())   # 20 districts for a 10-district table: a first warning

NameError: name 'raw' is not defined

In [ ]:
# 2 · Text columns: every distinct value (repr shows hidden spaces) · number columns: min and max
for col in ["district", "payment", "customer_type", "member_tier", "status"]:
    print(raw[col].map(repr).value_counts(dropna=False).to_string(), "\n")
print(raw.describe().round(1).to_string())             # look at min and max first
print(raw["basket_value_vnd"].head(3).tolist())         # a number column stored as text

In [ ]:
# 3 · Missing values, duplicates, and the two date formats
print(raw.isna().sum().to_string())
print("duplicate rows:", raw.duplicated().sum(), "| repeated order_id:", raw["order_id"].duplicated().sum())

slash = raw["order_time"].str.contains("/")
print("order_time written xx/xx/yyyy:", slash.sum(), "of", len(raw))
first = raw.loc[slash, "order_time"].str[:2].astype(int)
second = raw.loc[slash, "order_time"].str[3:5].astype(int)
print("first number > 12:", (first > 12).sum(), "| second number > 12:", (second > 12).sum(),
      "-> day first: dd/mm/yyyy")

odd = raw.loc[(raw["delivery_min"] <= 0) | (raw["delivery_min"] > 180), "delivery_min"]
print("impossible delivery times:", sorted(odd.tolist()))

**Problems found** (counts on the raw file, 5,973 rows)

| # | Problem | Column | How we found it |
|---|---|---|---|
| 1 | 60 exact duplicate rows: 60 `order_id` appear twice | all, `order_id` | `raw.duplicated().sum()` |
| 2 | 10 districts written 20 ways: extra space (`"Go Vap "`), capitals (`BINH THANH`), `Q1` / `Quận 1`, `Q7` / `Quận 7`, `Thu Duc` / `TP Thu Duc` (200 rows) | `district` | `nunique()`, `value_counts()` with `repr` |
| 3 | E-wallet written 4 ways: `E-wallet`, `e-wallet`, `ewallet`, `Momo/ZaloPay` (81 rows) | `payment` | `value_counts()` |
| 4 | Two date formats in one column: 45 rows `dd/mm/yyyy`, the rest `yyyy-mm-dd`; so the rows are not in time order either | `order_time` | `head()`, `str.contains("/")` |
| 5 | A number stored as text with commas (`"140,000"`) | `basket_value_vnd` | `info()` shows text, `head()` |
| 6 | Impossible delivery times: −12, −4, 0, 720, 999 and 1440 minutes | `delivery_min` | `describe()`: min and max |
| 7 | 26 basket values missing | `basket_value_vnd` | `isna().sum()` |
| 8 | Empty cells that are **not** errors: `member_tier` (4,164), `delivery_min` (312), `rating` (2,262). Task 2 explains why. | `member_tier`, `delivery_min`, `rating` | `isna().sum()`, then `pd.crosstab` in Task 2 |

## Task 2 · Name
**Dataset type:** a flat, static table (one row = one order), plus a lookup table of districts
(one row = one district), joined on `district` (many orders to one district).

| Column | Attribute type | Direction | Key / value | Notes on missing values |
|---|---|---|---|---|
| order_id | categorical (a name made of letters and digits) | – | **key** | none |
| order_time | quantitative (time) | sequential; its hour and weekday are cyclic | value | none |
| district | categorical | – | value (key to the district table) | none |
| customer_type | categorical (new / returning / member) | – (could be read as ordinal by loyalty) | value | none |
| member_tier | ordinal | sequential: Silver < Gold < Platinum | value | empty = **not applicable**: non-members |
| payment | categorical | – | value | none |
| items | quantitative (a count) | sequential | value | none |
| basket_value_vnd | quantitative (VND) | sequential | value | 25 **not recorded** (after removing duplicates) |
| status | categorical (delivered / cancelled) | – | value | none |
| delivery_min | quantitative (minutes) | sequential | value | empty = **not applicable**: cancelled orders; 6 impossible values set to missing |
| rating | ordinal (1–5) | sequential (or diverging around 3) | value | empty = **not given**: rating is optional |
| zone *(joined)* | ordinal | sequential by distance: Central < Inner < Outer | value | none |
| km_to_hub *(joined)* | quantitative (km) | sequential | value | none |

In [ ]:
# Key: one order_id per order?
print("order_id repeats in the raw file:", raw["order_id"].duplicated().sum(),
      "(the duplicate rows; unique after cleaning, checked in Task 3)")

# Missing values: the same reason for member_tier and rating?
print(pd.crosstab(raw["customer_type"], raw["member_tier"].isna().map({True: "tier empty", False: "tier set"})))
print(pd.crosstab(raw["status"], raw["rating"].isna().map({True: "no rating", False: "rating"})))
print(pd.crosstab(raw["status"], raw["delivery_min"].isna().map({True: "no time", False: "time"})))

# Ordered categories (zone and weekday follow in Task 4, once those columns exist)
TIERS = ["Silver", "Gold", "Platinum"]
RATINGS = [1, 2, 3, 4, 5]


def make_ordered(df):
    """member_tier and rating as ordered categories. Used again on the clean table in Task 4."""
    df = df.copy()
    df["member_tier"] = pd.Categorical(df["member_tier"], TIERS, ordered=True)
    df["rating"] = pd.Categorical(df["rating"], RATINGS, ordered=True)
    return df


named = make_ordered(raw)
print(named["member_tier"].cat.categories.tolist(), "| ordered:", named["member_tier"].cat.ordered)
print(named["rating"].cat.categories.tolist(), "| ordered:", named["rating"].cat.ordered)

**Answers to the three questions**

1. **Key:** `order_id`. In the raw file it repeats 60 times, exactly the 60 duplicate rows; after
   cleaning it is unique (the `assert` in Task 3 checks it).
2. **`member_tier` and `rating` are empty for different reasons.** `member_tier` is empty for exactly
   the non-members: *not applicable*, so we keep it empty and never invent a tier. `rating` is empty
   for every cancelled order and for about a third of delivered ones: *not given*, because rating is
   optional. We keep it empty and do not fill it; in Q3 we remember that customers who rate may not
   be typical.
3. **Ordered categories:** `member_tier` (Silver < Gold < Platinum) and `rating` (1 < … < 5),
   converted by `make_ordered()`; plus `zone` (Central < Inner < Outer) and `weekday` (Mon → Sun),
   created in Task 4.

## Task 3 · Clean
Write `clean_orders(df)` returning `(clean_df, log)`. Then justify every step in the Markdown cell below.

In [ ]:
DISTRICT_FIX = {"Q1": "District 1", "Quận 1": "District 1",      # after strip + title case
                "Q7": "District 7", "Quận 7": "District 7",
                "Thu Duc": "Thu Duc City", "Tp Thu Duc": "Thu Duc City"}
E_WALLET = {"e-wallet", "ewallet", "momo/zalopay"}               # Momo and ZaloPay are e-wallets
MAX_DELIVERY_MIN = 180      # 3 hours: generous; real deliveries take at most about 70 minutes
TEXT_COLS = ["order_id", "district", "customer_type", "member_tier", "payment", "status"]


def clean_orders(df):
    """Return (clean DataFrame, log dict). Each step is one decision."""
    log = {"rows_in": len(df)}
    df = df.copy()

    # 1 · dates: two formats in one column -> parse each format separately
    slash = df["order_time"].str.contains("/")                  # dd/mm/yyyy HH:MM
    time = pd.to_datetime(df["order_time"].where(~slash), format="%Y-%m-%d %H:%M")
    time[slash] = pd.to_datetime(df.loc[slash, "order_time"], format="%d/%m/%Y %H:%M")
    df["order_time"] = time
    log["dates_in_dd/mm/yyyy"] = int(slash.sum())

    # 2 · text: spaces, capitals and spelling variants
    district_before, payment_before = df["district"], df["payment"]
    for col in TEXT_COLS:
        df[col] = df[col].str.strip()
    df["district"] = df["district"].str.title().replace(DISTRICT_FIX)
    log["district_variants_fixed"] = int((df["district"] != district_before).sum())
    df.loc[df["payment"].str.lower().isin(E_WALLET), "payment"] = "E-wallet"
    log["payment_variants_fixed"] = int((df["payment"] != payment_before).sum())

    # 3 · a number stored as text: "140,000" -> 140000
    df["basket_value_vnd"] = pd.to_numeric(df["basket_value_vnd"].str.replace(",", ""))

    # 4 · duplicates (only after the text is fixed)
    log["duplicates_removed"] = int(df.duplicated().sum())
    df = df.drop_duplicates()

    # 5 · impossible delivery times: the order happened, the time is wrong -> missing
    bad = (df["delivery_min"] <= 0) | (df["delivery_min"] > MAX_DELIVERY_MIN)
    log["impossible_delivery_set_missing"] = int(bad.sum())
    df.loc[bad, "delivery_min"] = np.nan

    # 6 · missing values kept on purpose (nothing is changed, only counted)
    log["missing_basket_kept"] = int(df["basket_value_vnd"].isna().sum())
    log["no_tier_non_member"] = int(df["member_tier"].isna().sum())
    log["no_delivery_time_cancelled"] = int((df["status"] == "cancelled").sum())
    log["no_rating_kept"] = int(df["rating"].isna().sum())

    log["rows_out"] = len(df)
    return df.sort_values("order_time").reset_index(drop=True), log


clean, log = clean_orders(raw)
print(pd.Series(log).to_string())

In [ ]:
# Checks: what must be true after cleaning (an error here means a step went wrong)
assert clean["order_id"].is_unique                                # one row = one order
assert set(clean["district"]) == set(districts["district"])       # the 10 names of the table
assert clean["delivery_min"].dropna().between(0, MAX_DELIVERY_MIN).all()
assert (clean["member_tier"].isna() == (clean["customer_type"] != "member")).all()
assert clean.loc[clean["status"] == "cancelled", "delivery_min"].isna().all()
print(clean["district"].value_counts().to_string(), "\n")
print(clean["payment"].value_counts().to_string(), "\n")
print(clean.dtypes.to_string())

**Step-by-step justification** (rows counted on the raw file of 5,973 rows)

| # | What we did | Rows | Why |
|---|---|---|---|
| 1 | Parsed `order_time`, which mixes two formats: `yyyy-mm-dd` and `dd/mm/yyyy` | 45 in `dd/mm/yyyy` | 24 of the 45 start with a number above 12 and none has a month above 12, so they are day-first. Parsing each format separately loses no row. |
| 2 | Fixed `district` spellings (extra space, capitals, Q1 / Quận 1, Q7 / Quận 7, Thu Duc / TP Thu Duc) to the 10 names of the district table | 200 | Otherwise one district splits into several bars, and the merge in Task 4 cannot find their zone. |
| 3 | Merged `payment` variants (e-wallet, ewallet, Momo/ZaloPay) into `E-wallet` | 81 | Same method, different spelling. Judgement call: Momo and ZaloPay are e-wallet brands (24 rows). Without this, Q2 understates the e-wallet share. |
| 4 | Turned `basket_value_vnd` from text (`"140,000"`) into numbers | all rows | Text cannot be summed or averaged. |
| 5 | Removed exact duplicate rows, **after** fixing the text | 60 | Each was a second copy of the same `order_id`; they would count orders twice. Fixing text first means no copy survives under another spelling. |
| 6 | Set impossible `delivery_min` to missing: −12, −4, 0, 720, 999, 1440 | 6 | A delivery cannot take 0 minutes or a whole day; real ones take at most about 70 minutes. The orders did happen, so we keep the rows (district, payment, basket are still valid) and remove only the wrong time. |
| 7 | Kept missing `basket_value_vnd` | 25 | Not recorded. Deleting the orders would hide real orders; filling 0 would be false. Basket statistics skip them. |
| 8 | Kept empty `member_tier` | 4,123 | Not applicable: exactly the non-members. |
| 9 | Kept empty `delivery_min` of cancelled orders | 306 | Not applicable: these orders were never delivered. |
| 10 | Kept missing `rating` | 2,241 | Not given: rating is optional (and impossible for cancelled orders). Careful in Q3: customers who rate may not be typical. |

Result: 5,973 rows in, **5,913 orders** out.

## Task 4 · Reshape and combine

In [ ]:
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

# many_to_one: many orders per district, exactly one row per district in the table
orders = clean.merge(districts, on="district", how="left", validate="many_to_one")
print("rows before / after the merge:", len(clean), "/", len(orders))
print("orders without a zone:", orders["zone"].isna().sum())

orders["zone"] = pd.Categorical(orders["zone"], ["Central", "Inner", "Outer"], ordered=True)
orders["hour"] = orders["order_time"].dt.hour
orders["weekday"] = pd.Categorical(orders["order_time"].dt.strftime("%a"), DAYS, ordered=True)
orders["month"] = orders["order_time"].dt.month

print(orders[["order_time", "district", "zone", "km_to_hub", "hour", "weekday", "month"]].head())
print(orders.groupby("zone", observed=True)["km_to_hub"].agg(["min", "max", "size"]))

orders = make_ordered(orders)                                  # member_tier, rating (Task 2)
orders.to_csv(HERE / "gomart_orders_clean.csv", index=False)
print("saved:", HERE / "gomart_orders_clean.csv", orders.shape)

`orders` is the table for Task 5: one row per order, with its zone and distance to the hub,
plus `hour`, `weekday` (ordered Mon → Sun) and `month`. No order lost its zone in the merge.
It is also saved as `gomart_orders_clean.csv` next to this notebook. A CSV forgets types: after
`pd.read_csv` parse `order_time` again and re-create the ordered categories.
For averages of `rating`, use `orders["rating"].astype(float)`.

## Task 5 · From question to chart
### Q1 · Where are deliveries slow?
*Task (action + target): ...  Chart and why: ...*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pandas as pd

# Exclude cancelled orders for delivery-time comparisons.
# Resolve the dataframe name in case the notebook uses a different variable name
if "orders" not in globals():
    for name in ("orders_df", "data", "df"):
        if name in globals() and hasattr(globals()[name], "dropna"):
            orders = globals()[name]
            break
    else:
        # The notebook may store the dataframe name in a string variable, e.g. name == "df".
        data_name = globals().get("name")
        if isinstance(data_name, str) and data_name in globals() and hasattr(globals()[data_name], "dropna"):
            orders = globals()[data_name]
        else:
            # Use the notebook's actual dataframe name (e.g. "df"), or fall back to the common dataframe variable name.
            data_name = globals().get("name") or globals().get("data_name")
            if isinstance(data_name, str):
                orders = globals().get(data_name)
            else:
                orders = None

            if orders is None or not hasattr(orders, "dropna"):
                orders = globals().get("df")
            if orders is None or not hasattr(orders, "dropna"):
                # The notebook stores the main dataframe in a variable named by `name` / `data_name`
                if isinstance(name, str) and name in globals() and hasattr(globals()[name], "dropna"):
                    orders = globals()[name]
                elif isinstance(data_name, str) and data_name in globals() and hasattr(globals()[data_name], "dropna"):
                    orders = globals()[data_name]
                elif "orders_df" in globals() and hasattr(globals()["orders_df"], "dropna"):
                    orders = globals()["orders_df"]
                elif "data" in globals() and hasattr(globals()["data"], "dropna"):
                    orders = globals()["data"]
                elif "df" in globals() and hasattr(globals()["df"], "dropna"):
                    orders = globals()["df"]
                else:
                    raise NameError("The order dataframe is not defined. Load the data into a variable such as `orders` before running this cell.")

# Resolve the dataframe from notebook variables or the saved CSV fallback.
if orders is None or not hasattr(orders, "dropna"):
    for candidate in ("orders", "orders_df", "data", "df"):
        obj = globals().get(candidate)
        if hasattr(obj, "dropna"):
            orders = obj
            break
    else:
        candidate_name = globals().get("data_name") or globals().get("name")
        if isinstance(candidate_name, str):
            obj = globals().get(candidate_name)
            if hasattr(obj, "dropna"):
                orders = obj

    if orders is None or not hasattr(orders, "dropna"):

        csv_path = os.path.join(os.getcwd(), "gomart_orders_clean.csv")
        if os.path.exists(csv_path):
            orders = pd.read_csv(csv_path)
        else:
            # The CSV exists in the notebook directory; load it and keep a few
            # common dataframe aliases so later cells can resolve `orders`/`df`.
            # Search likely project/data directories instead of assuming the CSV sits in the notebook folder.
            search_roots = []
            if isinstance(DATA_DIR, str) and DATA_DIR:
                search_roots.append(DATA_DIR)
            search_roots.extend([
                os.getcwd(),
                os.path.dirname(os.getcwd()),
                os.path.abspath(os.path.join(os.getcwd(), "..")),
                os.path.abspath(os.path.join(os.getcwd(), "..", "..")),
            ])

            csv_path = None
            for root in dict.fromkeys(search_roots):
                if not root or not os.path.isdir(root):
                    continue

                direct_path = os.path.join(root, "gomart_orders_clean.csv")
                if os.path.isfile(direct_path):
                    csv_path = direct_path
                    break

                for current, _, files in os.walk(root):
                    if "gomart_orders_clean.csv" in files:
                        csv_path = os.path.join(current, "gomart_orders_clean.csv")
                        break
                if csv_path:
                    break

            if csv_path is None:
                raise FileNotFoundError(
                    "Could not find 'gomart_orders_clean.csv'. "
                    "Make sure the CSV is in the notebook folder or one of its parent/project folders."
                )

            orders = pd.read_csv(csv_path)
            globals()["orders"] = orders
            globals()["df"] = orders
            globals()["data"] = orders
            globals()["orders_df"] = orders
        if os.path.exists(csv_path):
            orders = pd.read_csv(csv_path)
            globals()["orders"] = orders
            globals()["df"] = orders
            globals()["data"] = orders
            globals()["orders_df"] = orders
        else:
            raise NameError(
            "The order dataframe is not defined. Load `orders` or make sure "
            "`gomart_orders_clean.csv` is in the notebook directory."
            )

# Normalize types expected by the later analysis.
if "order_time" in orders.columns:
    orders["order_time"] = pd.to_datetime(orders["order_time"], errors="coerce")
if "weekday" in orders.columns and not pd.api.types.is_categorical_dtype(orders["weekday"]):
    orders["weekday"] = pd.Categorical(
        orders["weekday"],
        categories=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
        ordered=True,
    )
if "rating" in orders.columns:
    orders["rating"] = orders["rating"].astype(float)

delivery_data = orders.dropna(subset=["delivery_min"]).copy()

if delivery_data.empty:
    raise ValueError("No orders with a non-missing delivery_min are available.")

zone_order = ["Central", "Inner", "Outer"]

# Find medians and ordering for the category comparisons.
zone_medians = (
    delivery_data.dropna(subset=["zone"])
    .groupby("zone", observed=True)["delivery_min"]
    .median()
    .reindex(zone_order)
    .dropna()
)
district_medians = (
    delivery_data.dropna(subset=["district"])
    .groupby("district", observed=True)["delivery_min"]
    .median()
    .sort_values()
)

if zone_medians.empty or district_medians.empty:
    raise ValueError("Completed orders must have zone and district values for the boxplots.")

slowest_zone = zone_medians.idxmax()
slowest_district = district_medians.idxmax()

# Highlight the slowest category in each plot.
zone_palette = {
    zone: ("#e68613" if zone == slowest_zone else "#b8c4ce")
    for zone in zone_order
}
district_palette = {
    district: ("#e68613" if district == slowest_district else "#4c78a8")
    for district in district_medians.index
}
district_order = district_medians.index.tolist()

# Compare delivery-time distributions across zones and districts.
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.boxplot(
    data=delivery_data.dropna(subset=["zone"]),
    x="zone",
    y="delivery_min",
    order=zone_order,
    hue="zone",
    hue_order=zone_order,
    palette=zone_palette,
    dodge=False,
    legend=False,
    ax=axes[0],
)
axes[0].set_title(f"Delivery times are typically longest in {slowest_zone}")
axes[0].set_xlabel("Delivery Zone")
axes[0].set_ylabel("Delivery Time (minutes)")

sns.boxplot(
    data=delivery_data.dropna(subset=["district"]),
    x="district",
    y="delivery_min",
    order=district_order,
    hue="district",
    hue_order=district_order,
    palette=district_palette,
    dodge=False,
    legend=False,
    ax=axes[1],
)
axes[1].set_title(f"{slowest_district} has the longest typical delivery time")
axes[1].set_xlabel("District")
axes[1].set_ylabel("Delivery Time (minutes)")
axes[1].tick_params(axis="x", rotation=45)

fig.tight_layout()
plt.show()

# Keep complete distance-time pairs for the scatter plot and correlation.
distance_data = delivery_data.dropna(subset=["km_to_hub"]).copy()

if distance_data.empty:
    raise ValueError("No completed orders have a non-missing km_to_hub value.")

pearson_r = distance_data["km_to_hub"].corr(distance_data["delivery_min"])
distance_var = distance_data["km_to_hub"].var()
can_fit_line = len(distance_data) >= 2 and distance_var > 0

# The OLS slope has units of delivery minutes per kilometer.
minutes_per_km = (
    distance_data["km_to_hub"].cov(distance_data["delivery_min"]) / distance_var
    if can_fit_line
    else float("nan")
)

# Plot distance against delivery time, colored by delivery zone.
fig, ax = plt.subplots(figsize=(9, 6))

sns.scatterplot(
    data=distance_data,
    x="km_to_hub",
    y="delivery_min",
    hue="zone",
    hue_order=zone_order,
    palette=zone_palette,
    alpha=0.7,
    ax=ax,
)

if can_fit_line:
    sns.regplot(
        data=distance_data,
        x="km_to_hub",
        y="delivery_min",
        scatter=False,
        color="black",
        line_kws={"linewidth": 2},
        ax=ax,
    )

if pearson_r > 0:
    distance_finding = "delivery time tends to increase with distance"
elif pearson_r < 0:
    distance_finding = "delivery time tends to decrease with distance"
elif pearson_r == 0:
    distance_finding = "there is no linear association with distance"
else:
    distance_finding = "the linear association with distance could not be estimated"

correlation_text = f" (Pearson r = {pearson_r:.2f})" if pearson_r == pearson_r else ""
ax.set_title(f"Distance relationship: {distance_finding}{correlation_text}")
ax.set_xlabel("Distance from Hub (km)")
ax.set_ylabel("Delivery Time (minutes)")
ax.legend(title="Delivery Zone")

fig.tight_layout()
plt.show()

print(
    f"Slowest typical zone: {slowest_zone} "
    f"(median {zone_medians[slowest_zone]:.1f} minutes); "
    f"slowest typical district: {slowest_district} "
    f"(median {district_medians[slowest_district]:.1f} minutes)."
)

if can_fit_line:
    print(
        f"Distance association: {distance_finding} "
        f"(Pearson r = {pearson_r:.2f}; {minutes_per_km:.2f} minutes per km)."
    )
else:
    print(f"Distance association: {distance_finding}; a trendline cannot be fit because distance does not vary.")

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Trực quan hóa dữ liệu\\DataViz_Day02_Group\\homework\\gh2_gomart\\gomart_orders_clean.csv'

In [ ]:
# TODO: write your code here

*Interpretation (one sentence):*

### Q2 · Is the move to e-wallet real?
*Task: ...  Chart and why: ...*

In [ ]:
# TODO: write your code here

*Interpretation:*

### Q3 · Do slow deliveries cost good ratings?
*Task: ...  Chart and why: ...*

In [ ]:
# TODO: write your code here

In [ ]:
# TODO: write your code here

*Interpretation:*

### Q4 · When are the peaks?
*Task: ...  Chart and why: ...*

In [ ]:
# TODO: write your code here

*Interpretation:*

## Task 7 · Team and AI
| Member | What they did | Share of work |
|---|---|---|
| | | |

In [ ]:
AI_USE = """
Tool(s) used:
What we asked:
What we checked or changed ourselves:
"""
print(AI_USE)